# Marco 5 - campanha FINAL (v7) no Colab

ESTADO: notebook escrito sem ter sido executado. Os arquivos v7 precisam estar COMMITADOS no GitHub com os nomes canonicos (rodar_campanha_marco5.py, sintonia_reativo.py, test_orquestrador_v7.py).

Ordem: 1 clonar, 2 conferir v7, 3 testes, 4 dry-run, 5 quick (opcional, mede tempo), 6 sintonia FINAL, 7 blocos 1 a 7, 8 latencia, 9 resumo.

Regras: pasta final = m5_final (NOVA e VAZIA; nao reaproveitar m5_saidas). Mesmos --norb 6, --norb-sint 6 e --seeds 10 11 12 13 14 em TODAS as celulas da campanha. A sessao do Colab pode cair: ao reabrir, rode de novo as celulas 1 e 2 e depois a celula do bloco que faltou (as partes prontas sao puladas; o orquestrador recusa config diferente da gravada).

In [ ]:
%%bash
cd /content
if [ -d mestrado ]; then cd mestrado && git pull; else git clone https://github.com/dafnhe-thor-barros/mestrado.git; fi
cd /content/mestrado/simulacoes/v2_auditado
git log --oneline | head -3
ls

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
grep -q campanha_config rodar_campanha_marco5.py && echo OK_orquestrador_v7 || echo FALTA_orquestrador_v7_COMMITAR
grep -q 120.0 sintonia_reativo.py && echo OK_sintonia_v7 || echo FALTA_sintonia_v7_COMMITAR
ls -la test_orquestrador_v7.py v2c.pt marco5_tems.py test_marco5.py
sha256sum marco5_tems.py rodar_campanha_marco5.py sintonia_reativo.py test_orquestrador_v7.py test_marco5.py v2c.pt
pip -q install pytest 2>&1 | tail -1
python -c 'import torch; print("torch", torch.__version__)'
nproc

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
mkdir -p m5_final
python -m pytest test_marco5.py test_orquestrador_v7.py -q 2>&1 | tee m5_final/pytest.txt | tail -40

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
python rodar_campanha_marco5.py --dry-run --norb 6 --seeds 10 11 12 13 14 --chunk 1 --norb-sint 6 --out m5_final

Celula 5 (OPCIONAL): teste rapido em pasta propria (m5_quick). Serve para ver se o fluxo roda e medir tempo. NAO usa m5_final.

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
rm -rf m5_quick
time python rodar_campanha_marco5.py --quick --jobs 2 --out m5_quick 2>&1 | tail -40

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
python rodar_campanha_marco5.py --resumo --out m5_quick 2>&1 | head -c 20000

Celula 6: sintonia FINAL (bloco 0), em segundo plano. Antes: confirme que m5_final NAO tem pasta parts nem sintonia_reativo.json (so pytest.txt pode existir).

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
ls m5_final
ls m5_final/parts 2>/dev/null && echo ATENCAO_parts_existe || echo parts_nao_existe_OK
setsid nohup python rodar_campanha_marco5.py --nova --blocos 0 --norb-sint 6 --norb 6 --seeds 10 11 12 13 14 --jobs 2 --chunk 1 --out m5_final > m5_final_bloco0.log 2>&1 &
sleep 5
cat m5_final_bloco0.log

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
pgrep -af rodar_campanha_marco5 | head -3 || echo nenhum_processo
ls -la m5_final
tail -n 15 m5_final_bloco0.log

Celula 7: blocos 1 a 7 em segundo plano (so depois de existir m5_final/sintonia_reativo.json). SEM --nova, com os MESMOS argumentos.

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
ls -la m5_final/sintonia_reativo.json
setsid nohup python rodar_campanha_marco5.py --blocos 1 2 3 4 5 6 7 --norb-sint 6 --norb 6 --seeds 10 11 12 13 14 --jobs 2 --chunk 1 --out m5_final > m5_final_blocos1a7.log 2>&1 &
sleep 5
cat m5_final_blocos1a7.log

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
pgrep -af rodar_campanha_marco5 | head -3 || echo nenhum_processo
for b in 1 2 3 4 5 6 7; do echo bloco $b: $(ls m5_final/parts/$b/*.json 2>/dev/null | wc -l) json, $(ls m5_final/parts/$b/*.divergiu 2>/dev/null | wc -l) divergiu; done
tail -n 12 m5_final_blocos1a7.log

Celula 8: latencia. Rode SOZINHA, sem nada mais em execucao (confira a celula de status: nenhum_processo). No Colab a CPU e compartilhada: cite sempre como 'Colab' e prefira repetir no servidor Debian (i5-8500T).

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
pgrep -af rodar_campanha_marco5 | head -3 || echo nenhum_processo_OK
python rodar_campanha_marco5.py --latencia --maquina 'Colab, CPU compartilhada' --out m5_final

Celula 9: resumo final para colar na conversa (rotule: campanha final, sintonia 6 orbitas, sementes 10 a 14). Se vier grande, cole em duas partes.

In [ ]:
%%bash
cd /content/mestrado/simulacoes/v2_auditado
python rodar_campanha_marco5.py --resumo --out m5_final
for f in m5_final/parts/4/pert_S1_*.err; do echo == $f; tail -n 2 $f; done

Celula 10 (OPCIONAL): copia de seguranca da pasta m5_final no Google Drive (a sessao do Colab apaga o disco). Pede autorizacao do Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/mestrado_backup
!cp -r /content/mestrado/simulacoes/v2_auditado/m5_final /content/drive/MyDrive/mestrado_backup/
!ls /content/drive/MyDrive/mestrado_backup/m5_final | head